# Notebook 04: Granger selection (Section 9.2, SOP 2)

Eligible candidates after G5: Science Garden rainfall (6-hourly) and daily RH, pressure, mean temperature (against the daily max of de-tided Pandacan). Steps: de-tide; ADF per training season (difference once if any p >= 0.05); lag matrices within seasons + season dummies; BIC lag (common sample); F-test, select if p < 0.05/m; ridge validation RMSE gain; correlation screen for the ablation. Sto. Niño and Fort Santiago: descriptive only (D2). Rules: `configs/granger.yaml`. Script: `python scripts/run_phase.py --phase 4`.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
from pasig import phase4
from pasig.gates import require_gates, read_gate
require_gates('G6', 'G9')

In [ ]:
out = phase4.run()

## Results (eligible candidates count toward m; stations are descriptive only)

In [ ]:
out['table']

In [ ]:
read_gate('G10')['decision'], out['h2']

## Ridge validation RMSE gain beside each p-value, and the correlation screen

In [ ]:
pd.DataFrame({k: {'gain_m': v['gain_m'], 'alpha': v['with']['alpha'], 'rmse_with': v['with']['val_rmse_mean'], 'rmse_without': v['without']['val_rmse_mean']} for k, v in out['gains'].items()}).T

In [ ]:
pd.DataFrame(out['xcorr']).T

## Model inputs written to `configs/selected_features.yaml`

In [ ]:
print(open('configs/selected_features.yaml').read()[:1500])

## Tests

In [ ]:
import subprocess
assert subprocess.call([sys.executable, '-m', 'pytest', '-q', 'tests/test_granger.py', 'tests/test_leakage.py']) == 0